# Sachsen-Anhalt municipal demographic pilot
Independent checks of the saved dataset and descriptive correlations. Rebuild raw extraction with `scripts/analyze_lsa_demographics.py`. The analysis is ecological, observational and preliminary.

In [1]:
from pathlib import Path
import json, pandas as pd, numpy as np
root = Path.cwd()
if not (root / 'data/2026-lsa').exists():
    root = next(p for p in root.parents if (p / 'data/2026-lsa').exists())
p = root / 'data/2026-lsa/reports/demographic-feasibility'
d = pd.read_csv(p / 'municipality_analysis.csv', dtype={'ags':str, 'kreis':str, 'census_ars':str, 'ars_2026q2':str})
r = pd.read_csv(p / 'party_correlations.csv')
assert len(d) == d.ags.nunique() == 218
assert d.valid_second_votes.sum() == 1315315
assert d.population_2025.sum() == 2120252
assert d.is_city.sum() == 104
assert (d.census_ars.str[:5] + d.census_ars.str[-3:] == d.ags).all()
assert (d.census_ars == d.ars_2026q2).all()
print('218 unique, complete municipal joins; 104 cities; vote/population totals match.')

218 unique, complete municipal joins; 104 cities; vote/population totals match.


In [2]:
coverage = pd.read_csv(p / 'feature_coverage.csv')
assert (coverage.n == 218).sum() == 11
assert d.abitur_share.notna().sum() == 54
assert d.ilo_unemployment_share.notna().sum() == 52
print(coverage[['label','n','missing']].to_string(index=False))
print('Education cohort fraction of valid votes:', d.loc[d.abitur_share.notna(),'valid_second_votes'].sum()/d.valid_second_votes.sum())

                          label   n  missing
          Population size (log) 218        0
       Population density (log) 218        0
         Residents aged 65+ (%) 218        0
       Residents aged 18–29 (%) 218        0
             Male residents (%) 218        0
  Population change 2024–25 (%) 218        0
     Non-German citizenship (%) 218        0
      One-person households (%) 218        0
   Owner-occupied dwellings (%) 218        0
           Vacant dwellings (%) 218        0
         Net cold rent (EUR/m²) 218        0
  Abitur/Fachhochschulreife (%)  54      164
No vocational qualification (%)  54      164
           ILO unemployment (%)  52      166
   Manufacturing employment (%)  54      164
Education cohort fraction of valid votes: 0.7068261214994127


In [3]:
official = json.loads((root / 'data/2026-lsa/reports/git-timeline/80fa3052/latest_official_rows.json').read_text())
parties = json.loads((p / 'party_names.json').read_text())
land = official['lsa:LAND:15:TOTAL']
for party in parties:
    assert d[party + '_votes'].sum() == land['parties'][party]
    assert np.allclose(d[party + '_share'], 100*d[party + '_votes']/d.valid_second_votes)
assert np.allclose(d[[c+'_share' for c in parties]].sum(axis=1),100)
print('All 15 party numerators and every valid-vote denominator reconcile.')

All 15 party numerators and every valid-vote denominator reconcile.


In [4]:
for row in r.itertuples():
    sample = d[[row.feature, row.code+'_share']].dropna()
    assert len(sample) == row.n
    independently_computed = sample.rank(method='average').corr().iloc[0,1]
    assert abs(independently_computed-row.spearman)<1e-12
print('All 225 Spearman values independently reproduced from saved source joins.')
print(r[(r.feature=='abitur_share') & r.party.isin(['AfD','GRÜNE'])][['party','n','spearman']].to_string(index=False))

All 225 Spearman values independently reproduced from saved source joins.
party  n  spearman
  AfD 54 -0.587193
GRÜNE 54  0.757500


## Interpretation
The cohort includes every municipality, but education/employment comparisons use the published subset only. The study does not infer individual party preferences or causal effects. The report and CSV include sensitivity to the three independent cities, city status, district exclusions, weighting and within-district ranks. Follow-up models must deal with correlated features, prior party support, compositional shares, spatial dependence and source suppression.